In [ ]:
import h5py
import cv2
import numpy as np
import random
from pathlib import Path

In [ ]:
def create_h5_dataset(sample_rate_empty = 0.2, seed = 77): #AK77
    image_folder = Path('preprocessing_output')
    mask_folder = Path('masks_output')
    output_h5 = Path('cell_dataset.h5')

    for folder in [image_folder, mask_folder]:
        if not folder.exists():
            print(f"{folder} not exist")
            return

    random.seed(seed)
    np.random.seed(seed)

    #樣本分類(positive, empty)
    all_image_paths = sorted(list(image_folder.glob('*.png'))) #路徑.glob(檔名) 搜索路徑中所有帶有指定檔名內容的檔案
    positive_samples = []
    empty_samples = []

    for img_path in all_image_paths:
        mask_path = mask_folder / img_path.name
        if mask_path.exists():
            mask_data = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) #cv2.imread(path, flag) #cv2.IMREAD_GRAYSCALE雙通道灰階 #讀出來會是一個numpy matrix
            if np.sum(mask_data) == 0:
                empty_samples.append(img_path)
            else:
                positive_samples.append(img_path)
    print(f"sorting complete: {len(positive_samples)+len(empty_samples)}")

    num_empty_keep = int(len(positive_samples) * sample_rate_empty)
    num_empty_keep = min(num_empty_keep, len(empty_samples))
    selected_empty = random.sample(empty_samples, num_empty_keep) if empty_samples else [] #random.sample(清單, 數量)不重複的隨機抽取

    valid_samples = positive_samples + selected_empty #合併成最終list

    random.shuffle(valid_samples) #random.shuffle(list)列表中隨機洗牌 #EveryDayImShuffling

    if not valid_samples:
        print(".png file not exist")
        return

    print(f"cell positive: {len(positive_samples)}")
    print(f"cell negative: {len(selected_empty)}")